# Different Metrics for evaluating ML Model:
1. Accuracy: $$\frac{NumCorrectPred}{Total}$$
> * Nice with balanced class data (e.g. equal amount of Zeros and Ones for binomial category)
2. Recall: $$\frac{TruePositivesPred}{TotalPositiveActual}$$
> * Measures ability to find all relevant cases
3. Precision: $$\frac{TruePosPred}{TotalPosPred}$$
> * Measure of ability to identify ONLY relevant data points
4. F1: $$F_{1} = 2 * \frac{precision * recall}{precision + recall}$$
> * Harmonic mean of precision and recall such that a balance of the two can be viewed
> * Harmonic mean punishes extreme values
> > * e.g. Precision = 1, Recall = 0, Mean = 0.5, F1=0

# Confusion Matrix
<img src="Confusion_Matrix.png">

<img src="Confusion_Matrix_w_Metrics.png">

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('smsspamcollection.tsv',sep='\t')

In [ ]:
df.head()

In [ ]:
# No missing values
df.isnull().sum()

In [ ]:
len(df)

In [ ]:
df['label'].unique()

In [ ]:
df['label'].value_counts()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
sns.pairplot(df[["length", "label"]], hue="label", size=5)
plt.legend(('ham','spam'))
plt.show()

In [ ]:
sns.pairplot(df[["punct", "label"]], hue="label", size=5)
plt.legend(('ham','spam'))
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
# X is the feature data
X = df[['length', 'punct']]
# y is our label/target data
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr_mod = LogisticRegression(solver='lbfgs')

In [ ]:
lr_mod.fit(X_train, y_train)

In [ ]:
prediction = lr_mod.predict(X_test)
lr_mod.predict(X_test)

In [ ]:
# Accuracy
sum(lr_mod.predict(X_test) == y_test) / len(y_test)

In [ ]:
from sklearn import metrics

In [ ]:
display(metrics.confusion_matrix(y_test,prediction))

In [ ]:
conf_df = pd.DataFrame(metrics.confusion_matrix(y_test,prediction), 
                            index=['TrueHam','TrueSpam'], 
                            columns=['PredHam','PredSpam'])
conf_df

In [ ]:
print(metrics.classification_report(y_test,prediction))

In [ ]:
print(metrics.accuracy_score(y_test,prediction)) 

In [ ]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

prediction = nb_model.predict(X_test)

display(pd.DataFrame(metrics.confusion_matrix(y_test,prediction), 
                            index=['TrueHam','TrueSpam'], 
                            columns=['PredHam','PredSpam']))

print(metrics.classification_report(y_test,prediction))

In [ ]:
from sklearn.svm import SVC

svc_model = SVC(gamma='auto')

svc_model.fit(X_train, y_train)

prediction = svc_model.predict(X_test)

display(pd.DataFrame(metrics.confusion_matrix(y_test,prediction), 
                            index=['TrueHam','TrueSpam'], 
                            columns=['PredHam','PredSpam']))

print(metrics.classification_report(y_test,prediction))

# Feature Extraction from Text

### Count Vectorization

In [ ]:
messages = ["Hey, lets go to the game today!",
            "Call your sister.",
            "Want to go walk your dogs?"]

from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer()

vect.fit_transform(messages)
vect.get_feature_names()

### Term Frequency - Inverse Document Frequency
* Logarithmically scaled inverse fraction of the documents that contain the word
> Obtained by dividing the total number of documents by the number of documents containing the term, and then taking the logarithm of that quotient

$$tf(t,d) =  \frac{NumTimesTermInDoc}{NumTermsInDoc}$$

$$idf(t,D)=log\left(\frac{NumDoc}{NumDocWithTerm}\right)$$

$$tfidf(t,d,D) = tf(t,d) \cdot idf(t,D) $$

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('smsspamcollection.tsv', sep='\t')

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df['label'].value_counts()

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = df['message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
count_vec = CountVectorizer()

In [ ]:
# Steps:
# 1. Fit the vectorizer to the data (build the vocab, count the number of words)
count_vec.fit(X_train)
X_train_counts = count_vec.transform(X_train)
# 2. Transform the original text message to a vector
X_train_counts = count_vec.transform(X_train)

# OR Both at once:
X_train_counts = count_vec.fit_transform(X_train)

In [ ]:
X_train_counts

In [ ]:
X_train.shape

In [ ]:
X_train_counts.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfTransformer

In [ ]:
tfidf_trans = TfidfTransformer()

X_train_tfidf = tfidf_trans.fit_transform(X_train_counts)

In [ ]:
X_train_tfidf.shape

In [ ]:
# We can also combine those steps:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

X_train_tfidf = vectorizer.fit_transform(X_train)

In [ ]:
X_train_tfidf.shape

In [ ]:
from sklearn.svm import LinearSVC

clf = LinearSVC()

clf.fit(X_train_tfidf, y_train)

In [ ]:
# Utilizing pipeline to perform this task:
from sklearn.pipeline import Pipeline

In [ ]:
text_clf = Pipeline([('tfidf', TfidfVectorizer()),
                     ('clf', LinearSVC()
                     )
                    ])

In [ ]:
text_clf.fit(X_train,y_train)

In [ ]:
predictions = text_clf.predict(X_test)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

In [ ]:
display(pd.DataFrame(metrics.confusion_matrix(y_test,predictions), 
                            index=['TrueHam','TrueSpam'], 
                            columns=['PredHam','PredSpam']))

print(metrics.classification_report(y_test,predictions))

print(metrics.accuracy_score(y_test, predictions))

In [ ]:
text_clf.predict(["Hi how are you doing today"])

In [ ]:
text_clf.predict(["CONGRATULATIONS! You've been selected as a winner. TEXT WON to 44255 congratulations free entry to contest to contest"])

# Text Classification Project